In [1]:
from pathlib import Path
import sys
import polars as pl
import plotly.express as px
import pandas as pd

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts.rq3_function_lib import show_median_price_heatmap_per_region, mann_whitney_test_border_prices, show_border_price_difference, perform_matched_panel_regression_autobahn_stations
from scripts.rq3_function_lib import plot_autobahn_premium_boxplot, plot_yearly_autobahn_premium_line, plot_autobahn_premium_histogram, plot_station_price_map, plot_autobahn_premium_barchart
from scripts.rq3_function_lib import perform_wilcoxon_variance_test_on_autobahn

# Regional price differences and price stability

In [2]:
region_price_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/regions_avg_prices_per_year')
region_path = Path(r'/Users/sebastian/data-science-projekt/plz_leitregionen.csv')


In [3]:
year = 2022
fuel_type = "e10"

In [4]:
show_median_price_heatmap_per_region(region_price_path, region_path, year, fuel_type)

How does the price at the stations close to the german border (<=15km dist) differ from other stations in their surrounding area?

In [5]:
border_stations_file = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/lower_border_stations.csv')
border_stations = pl.read_csv(border_stations_file, schema_overrides = {"post_code": pl.Utf8})

print (border_stations.head())

shape: (5, 7)
┌──────────────────┬───────────┬───────────┬─────────────────┬───────────┬─────────────────┬───────┐
│ uuid             ┆ latitude  ┆ longitude ┆ neighbour_count ┆ dist_km   ┆ border_region   ┆ brand │
│ ---              ┆ ---       ┆ ---       ┆ ry              ┆ ---       ┆ ---             ┆ ---   │
│ str              ┆ f64       ┆ f64       ┆ ---             ┆ f64       ┆ str             ┆ str   │
│                  ┆           ┆           ┆ str             ┆           ┆                 ┆       │
╞══════════════════╪═══════════╪═══════════╪═════════════════╪═══════════╪═════════════════╪═══════╡
│ 005056ba-7cb6-1e ┆ 50.79344  ┆ 6.47057   ┆ Belgium         ┆ 22.161005 ┆ Surrounding     ┆ STAR  │
│ d2-bceb-9c6b14…  ┆           ┆           ┆                 ┆           ┆ (8-25km)        ┆       │
│ 14853398-0aff-41 ┆ 53.07093  ┆ 14.25534  ┆ Poland          ┆ 5.495808  ┆ Border (0-8km)  ┆ Shell │
│ f4-8297-ecaa23…  ┆           ┆           ┆                 ┆           ┆   

In [6]:
fig = px.scatter_map(border_stations,
                    lat = "latitude",
                    lon= "longitude",
                    color = "border_region",
                    hover_name = "neighbour_country",
                    hover_data = "neighbour_country",
                    center = {"lat": 51.16, "lon": 10.45},
                    zoom = 4,
                    map_style = "open-street-map",
                    title = "Border and surrounding stations in germany")

fig.update_layout(margin = {"r":0,"t":50,"l":0,"b":0})
fig.update_traces(marker = dict(size = 15, opacity = 1))
fig.show()

In [7]:
test_df = pl.read_parquet(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/station_daily_mean_and_median_by_month/2025/2025-12.parquet')

print(test_df.head(10))
print(test_df.columns)

shape: (10, 9)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬──────────┐
│ station_uu ┆ day       ┆ diesel_me ┆ diesel_me ┆ … ┆ e5_median ┆ e10_mean ┆ e10_media ┆ n_events │
│ id         ┆ ---       ┆ an        ┆ dian      ┆   ┆ ---       ┆ ---      ┆ n         ┆ ---      │
│ ---        ┆ date      ┆ ---       ┆ ---       ┆   ┆ f64       ┆ f64      ┆ ---       ┆ u32      │
│ str        ┆           ┆ f64       ┆ f64       ┆   ┆           ┆          ┆ f64       ┆          │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪══════════╡
│ 00060075-0 ┆ 2025-11-3 ┆ 1.609     ┆ 1.609     ┆ … ┆ 1.709     ┆ 1.649    ┆ 1.649     ┆ 1        │
│ 001-4444-8 ┆ 0         ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ 888-acdc00 ┆           ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ …          ┆           ┆           ┆           ┆   ┆           ┆          

Mann-Whitney-U-Test for all years and for each year, calculated seperatly for each bordering country

TODO: genaue mathematische erklärung & formeln etc.

In [8]:
median_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/station_daily_mean_and_median_by_month')

mann_whitney_test_border_prices(median_path, border_stations_file, "diesel")

=== absolute results (over all years) ===


,Country,N_border,N_surrounding,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%)
0,Belgium,34,62,1.309,1.299,0.010,2.331846e-01,False
1,France,171,366,1.329,1.317,0.012,8.538706e-07,True
2,Poland,40,60,1.300,1.299,0.001,8.735863e-01,False
3,Denmark,33,21,1.279,1.279,0.000,6.085410e-01,False
4,Czechia,55,237,1.289,1.299,-0.010,3.238376e-01,False
5,Netherlands,257,397,1.309,1.299,0.010,2.954649e-06,True
6,Switzerland,54,54,1.349,1.326,0.023,4.726668e-03,True
7,Austria,91,242,1.339,1.329,0.010,6.066301e-05,True



=== yearly result (excerpt) ===


,year,Country,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%),N_border,N_surrounding
0,2014,Belgium,1.349,1.344,0.005,0.071238,False,33,51
1,2015,Belgium,1.169,1.164,0.005,0.183879,False,33,52
2,2016,Belgium,1.094,1.084,0.010,0.025023,True,33,54
3,2017,Belgium,1.169,1.159,0.010,0.005420,True,33,58
4,2018,Belgium,1.289,1.279,0.010,0.010717,True,34,61
5,2019,Belgium,1.269,1.259,0.010,0.006764,True,33,56
6,2020,Belgium,1.089,1.079,0.010,0.146940,False,33,55
7,2021,Belgium,1.379,1.369,0.010,0.208867,False,31,54
8,2022,Belgium,1.959,1.959,0.000,0.186061,False,31,51
9,2023,Belgium,1.689,1.689,0.000,0.181195,False,31,51


In the above test, there could be external effect like stations that are on the autobahn, which could distort the test result.
Therefore, we're now filtering out all stations from our df that are on the autobahn and then perform the test again to see how it changes.

In [9]:
non_autobahn_border_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/lower_non_autobahn_border_stations.csv')
mann_whitney_test_border_prices(median_path, non_autobahn_border_path, "diesel")

=== absolute results (over all years) ===


,Country,N_border,N_surrounding,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%)
0,France,160,364,1.329,1.314,0.015,0.000006,True
1,Belgium,34,62,1.309,1.299,0.010,0.233185,False
2,Czechia,55,230,1.289,1.299,-0.010,0.479434,False
3,Denmark,33,21,1.279,1.279,0.000,0.608541,False
4,Switzerland,54,53,1.349,1.329,0.020,0.005640,True
5,Netherlands,255,393,1.309,1.299,0.010,0.000001,True
6,Poland,39,58,1.301,1.299,0.002,0.853299,False
7,Austria,86,236,1.339,1.329,0.010,0.000188,True



=== yearly result (excerpt) ===


,year,Country,Median_border,Median_surrounding,Price_difference,p_value,Significant (5%),N_border,N_surrounding
0,2014,France,1.349,1.349,0.000,1.226814e-02,True,148,337
1,2015,France,1.179,1.174,0.005,3.543328e-03,True,154,344
2,2016,France,1.099,1.089,0.010,1.351552e-03,True,155,347
3,2017,France,1.179,1.164,0.015,8.043702e-06,True,159,358
4,2018,France,1.323,1.304,0.019,6.316942e-10,True,160,353
5,2019,France,1.289,1.274,0.015,4.220023e-06,True,160,348
6,2020,France,1.094,1.082,0.012,5.985612e-03,True,160,344
7,2021,France,1.369,1.364,0.005,6.111697e-01,False,159,337
8,2022,France,1.974,1.969,0.005,6.429403e-01,False,156,330
9,2023,France,1.709,1.709,0.000,6.120821e-01,False,153,324


To visualize the (missing) price difference we plot the median prices for a year and for a specific border region.

In [10]:
year = 2022
country = "Netherlands"
 

show_border_price_difference(median_path, non_autobahn_border_path, "diesel", country, year)

Now we check if the border region non autobahn stations are brand stations or non brand stations

In [11]:

stations_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/stations.csv')
stations_df = pl.read_csv(stations_path, schema_overrides={"post_code": pl.Utf8})
no_autobahn_border_df = pl.read_csv(non_autobahn_border_path, schema_overrides = {"post_code": pl.Utf8})

brand_border_region_df = (stations_df.join(no_autobahn_border_df,
                                           how = "inner",
                                           on = "uuid"))

In [12]:

premium_brands = ["ARAL", "SHELL", "JET", "TOTAL", "TOTAL ENERGIES", "ESSO", "AVIA",
    "HEM", "HOYER", "ORLEN", "Q1", "STAR", "RAIFFEISEN", "AGIP",
    "ENI", "OMV", "OIL!", "WESTFALEN"]
#categorize each station into brand or non brand
brand_df = (brand_border_region_df.with_columns(
    pl.col("brand").str.to_uppercase().str.strip_chars().alias("clean_brands")
).with_columns(
    pl.when(pl.col("clean_brands").is_in(premium_brands)).then(pl.lit("brand"))
    .otherwise(pl.lit("non brand")).alias("brand_category")
))
#aggregate and calculate percents
percent_df = (brand_df.group_by(["border_region", "brand_category"])
              .agg(pl.len().alias("counter"))
              .with_columns((pl.col("counter") / pl.col("counter").sum().over("border_region") * 100)
                            .round(2).alias("percentage")))
pivot_df = (percent_df.pivot(on = "brand_category",
                             index = "border_region",
                             values = "percentage"))

print("=== brand structure: border vs. surrounding regions ===")
print(pivot_df)

=== brand structure: border vs. surrounding regions ===
shape: (2, 3)
┌──────────────────────┬───────────┬───────┐
│ border_region        ┆ non brand ┆ brand │
│ ---                  ┆ ---       ┆ ---   │
│ str                  ┆ f64       ┆ f64   │
╞══════════════════════╪═══════════╪═══════╡
│ Surrounding (8-25km) ┆ 38.69     ┆ 61.31 │
│ Border (0-8km)       ┆ 38.42     ┆ 61.58 │
└──────────────────────┴───────────┴───────┘


In [13]:
fig_border_brand = px.scatter_map(brand_border_region_df,
                    lat = "latitude",
                    lon= "longitude",
                    color = "brand",
                    hover_name = "brand",
                    hover_data = "post_code",
                    center = {"lat": 51.16, "lon": 10.45},
                    zoom = 4,
                    map_style = "open-street-map",
                    title = "brands of border stations")

fig_border_brand.update_layout(margin = {"r":0,"t":50,"l":0,"b":0})
fig_border_brand.update_traces(marker = dict(size = 15, opacity = 1))
fig_border_brand.show()

The next part of the questions looks for the price differences between autobahn stations and regular stations. 
For that we use a matched panel regression that absorbs fixed regional and time effects.

We iterate over each fuel type, first performing the test on the mean prices and then to check the results, again over the median prices.

In [ ]:
autobahn_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/autobahn_stations.csv')

fuel_types = ["diesel", "e5", "e10"]
statistics = ["mean", "median"]

test_summaries = []
analysis_panel_list = []
residuals_list = []

for fuel in fuel_types:
    for stat in statistics:
        
        res, panel, residuals = perform_matched_panel_regression_autobahn_stations(median_path,stations_path,autobahn_path,fuel,stat,return_residuals=True)
        test_summaries.append(res)
        analysis_panel_list.append(panel)
        residuals_list.append(residuals)



residual_df = pd.concat(residuals_list, ignore_index=True)
summary_df = pd.concat(test_summaries, ignore_index = True)
print("\nSummary:")
summary_df



Summary:


,year,fuel_type,statistic,autobahn_coef,standard_error,p_value,ci_low,ci_high,n_observed
0,2014,diesel,mean,0.043297,0.001647,0.0,0.040070,0.046525,377130
1,2015,diesel,mean,0.051106,0.001983,0.0,0.047219,0.054994,685629
2,2016,diesel,mean,0.068193,0.002751,0.0,0.062802,0.073584,707278
3,2017,diesel,mean,0.080873,0.003013,0.0,0.074967,0.086778,716043
4,2018,diesel,mean,0.121318,0.003777,0.0,0.113914,0.128721,720505
...,...,...,...,...,...,...,...,...,...
73,2022,e10,median,0.215793,0.008074,0.0,0.199968,0.231618,617169
74,2023,e10,median,0.277823,0.010266,0.0,0.257703,0.297944,609842
75,2024,e10,median,0.287634,0.012263,0.0,0.263598,0.311670,603384
76,2025,e10,median,0.302458,0.013231,0.0,0.276526,0.328391,598113


In [22]:
#join the panel dfs into one df
keys = ["match_set_uuid", "station_uuid", "autobahn", "dist_km", "year", "latitude", "longitude", "date", "brand", "brand_category"]

# 1) build base table from shared columns
base_df = (
    pl.concat([df.select(keys) for df in analysis_panel_list], how="vertical")
    .unique()
)

# 2) add the one new metric column from each panel
panel_df = base_df

for df in analysis_panel_list:
    # identify the one non-key column
    value_cols = [c for c in df.columns if c not in keys]

    if len(value_cols) != 1:
        raise ValueError(f"Expected exactly 1 value column, got {value_cols}")

    value_col = value_cols[0]

    panel_df = panel_df.join(
        df.select(keys + [value_col]).unique(subset=keys),
        on=keys,
        how="left"
    )

panel_df.head()
print(panel_df.shape)

(7934988, 16)


plotting the results

In [16]:
plot_yearly_autobahn_premium_line(summary_df)

#TODO:fix plot so that it looks ok

In [17]:
plot_autobahn_premium_boxplot(summary_df)

In [18]:
plot_autobahn_premium_barchart(summary_df)

#TODO: fic plot, currently for each fuel type the mean and median value is summed up. each bar is one half mean and the other median

In [19]:
plot_autobahn_premium_histogram(summary_df)

the station prices in germany

In [20]:
plot_station_price_map(panel_df, "diesel","median")

Now we perform a wilcoxon variance test to see wether the autobahn station prices differ from normal station prices in variance. we used the residual error to calculate this, so the regional and time effects are filtered out.

In [28]:
wilcoxon_df = perform_wilcoxon_variance_test_on_autobahn(residual_df)

print(wilcoxon_df.head(15))

    year measure  n_pairs  mean_test_volatility  mean_control_volatility  \
0   2014     mad      294              0.011654                 0.011069   
1   2015     mad      299              0.015624                 0.011839   
2   2016     mad      301              0.014081                 0.011854   
3   2017     mad      307              0.014609                 0.012092   
4   2018     mad      374              0.019972                 0.016912   
5   2019     mad      309              0.018834                 0.013823   
6   2020     mad      305              0.019792                 0.014994   
7   2021     mad      298              0.017602                 0.013051   
8   2022     mad      296              0.036291                 0.021432   
9   2023     mad      300              0.030703                 0.017223   
10  2024     mad      241              0.030864                 0.015777   
11  2025     mad      236              0.019933                 0.014305   
12  2026    